# Session 08: Reasoning Trajectories

**Student implementer and maintainer:** YQ-HA. This notebook records code typed,
executed, and adapted while following the instructor during class, together with
the student's post-course completions and maintenance.

**Course:** Tyler Shoemaker, [Introduction to Interpretability for Language Models](https://digitaltheorylab.github.io/2026_interpretability-reader/).
Reference implementation: [08_reasoning.md](https://github.com/digitaltheorylab/2026_interpretability-reader/blob/main/chapters/08_reasoning.md).

**Repair assistance:** OpenAI Codex assisted with locating and explaining errors,
restoring omitted course code, and checking coverage and runnability. `Repair:`
comments identify corrected mistakes; `Course completion:` comments identify
restored steps. Original classroom notes, including `##` comments, are retained.

Run from the first cell with the kernel in `notebooks/`. New results go to
`results/session-08/`. This repaired version awaits a new Colab run.


In [ ]:
from pathlib import Path
import json
import platform
import sys
import matplotlib.pyplot as plt
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "session-08"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FACET_HEIGHT = 4
FACET_ASPECT = 1.2
def save_figure(name):
    plt.gcf().savefig(RESULTS_DIR / name, bbox_inches="tight", dpi=150)
    plt.show()
    plt.close("all")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from scipy.signal import find_peaks
from scipy.stats import median_abs_deviation
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Repair: Use T4-compatible precision; the saved original run did not leave a usable model.
checkpoint = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(
    checkpoint, device_map="auto", dtype=torch.float16
)

In [ ]:
torch.manual_seed(5167)
torch.set_grad_enabled(False)
model.eval()

In [ ]:
# Course completion: Restore generation, trace extraction, and predictive-state alignment before the coherence notes.
prompt = (
    "A store has 40 apples. In the morning it sells a third of them. "
    "In the afternoon a delivery adds 18 more apples, and then 25% of the "
    "apples in the store are sold. How many apples are left at closing? "
    "Reason briefly, then give the answer."
)


In [ ]:
# Course completion: Restore generation, trace extraction, and predictive-state alignment before the coherence notes.
messages = [{"role": "user", "content": prompt}]
chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)


In [ ]:
# Course completion: Restore generation, trace extraction, and predictive-state alignment before the coherence notes.
inputs = tokenizer(chat, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=2048,
    do_sample=True,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    return_dict_in_generate=True,
)


In [ ]:
# Course completion: Restore generation, trace extraction, and predictive-state alignment before the coherence notes.
sequence = outputs.sequences[0]
new_ids = sequence[inputs["input_ids"].shape[-1]:]
ids = new_ids.tolist()


In [ ]:
# Course completion: Restore generation, trace extraction, and predictive-state alignment before the coherence notes.
think_open = tokenizer.convert_tokens_to_ids("<think>")
think_close = tokenizer.convert_tokens_to_ids("</think>")


In [ ]:
# Course completion: Restore generation, trace extraction, and predictive-state alignment before the coherence notes.
if think_open in ids:
    start = ids.index(think_open) + 1
else:
    start = 0

if think_close in ids:
    print("OK: reasoning completed")
    end = ids.index(think_close)
else:
    print("WARNING: met token budget but didn't finish reasoning")
    end = len(ids)


In [ ]:
# Course completion: Restore generation, trace extraction, and predictive-state alignment before the coherence notes.
cot_ids = new_ids[start:end]


In [ ]:
# Course completion: Restore generation, trace extraction, and predictive-state alignment before the coherence notes.
prompt_len = inputs["input_ids"].shape[-1]


In [ ]:
# Course completion: Restore generation, trace extraction, and predictive-state alignment before the coherence notes.
with torch.no_grad():
    full_outputs = model(sequence[:-1].unsqueeze(0), output_hidden_states=True, use_cache=False)
num_layer = len(full_outputs.hidden_states)
LAYER = 1 + (num_layer - 2) // 2  # Transformer state, excluding embedding index 0.
print("Hidden-state tuple index:", LAYER)


In [ ]:
# Course completion: Restore generation, trace extraction, and predictive-state alignment before the coherence notes.
# Hidden state at position t predicts generated token t+1.
# Shift by one so each trace token is aligned to the state that predicted it.
trajectory = full_outputs.hidden_states[LAYER][0].float().cpu()
cot_slice = slice(prompt_len + start - 1, prompt_len + end - 1)


In [ ]:
# Course completion: Restore generation, trace extraction, and predictive-state alignment before the coherence notes.
hidden = trajectory[cot_slice]


In [ ]:
# Course completion: Restore generation, trace extraction, and predictive-state alignment before the coherence notes.
cot_ids = cot_ids[:hidden.shape[0]]


In [ ]:
# what constitute a reasoniong step in CoT? Is it an individual token?
# Probably not. More than likely, a reasoning step is multiple tokens joined together by
# some means. How can we extract multi-token sequences from the model outputs
# that get us to this meso-level of "reasoning step" that apparently characterizes
# chain-of-thought?
#
# Finding steps
#
# Estimate coherence length, which is the timescale of a generation
# Use autocorrelation
#
# 1 2 3 4 5 6 7 8 9               - lag 1
#   1 2 3 4 5 6 7 8 9             - lag 2
#       1 2 3 4 5 6 7 8 9         - lag 3
#             1 2 3 4 5 6 7 8 9   - lag 4


In [ ]:
# Course completion: Restore the coherence-length implementation inline.
def coherence_length(hidden, max_lag=50, eps=1e-8):
    """Estimate the trajectory's intrinsic timescale.

    Parameters
    ----------
    hidden : torch.Tensor
        Trajectory hidden states, shape (num_token, num_dim)
    max_lag : int
        Maximum lag time (in tokens) we allow for a timescale
    eps : float
        Epsilon value for numerical stability

    Returns
    -------
    tuple[int, torch.Tensor]
        - tau: the lag at which the autocorrelation of the hidden-state
          trajectory decays to 1/e of its lag-1 value
        - autocorr: the full curve across all lags
    """
    # Subtract the mean direction to remove the model's common component, which
    # otherwise inflates every similarity toward a high floor and masks actual
    # decay signal
    h = hidden - hidden.mean(dim=0, keepdim=True)
    h = F.normalize(h, dim=-1, eps=eps)

    # Set our maximum lag: minimum beween `max_lag` and the number of tokens
    max_lag = min(max_lag, h.shape[0] - 1)

    # Run autocorrelation
    autocorr = []
    for d in range(1, max_lag + 1):
        ac = (h[:-d] * h[d:]).sum(-1).mean()
        autocorr.append(ac)

    autocorr = torch.stack(autocorr)

    # Set our baseline decay and compute crossings: where `autocorr` dips below
    # `target`
    target = autocorr[0] / np.e
    crossings = (autocorr < target).nonzero(as_tuple=False)

    # `tau` is the lag-1 value where values in `autocorr` decay below `target`
    if len(crossings):
        tau = crossings[0].item() + 1
    else:
        tau = max_lag

    return tau, autocorr


In [ ]:
# Course completion: Restore coherence-window estimation and its plot.
WINDOW, autocorr = coherence_length(hidden)
print("Window size for chunking tokens:", WINDOW)


In [ ]:
# Course completion: Restore coherence-window estimation and its plot.
fig, ax = plt.subplots()
lag = np.arange(1, len(autocorr) + 1)
sns.lineplot(x=lag, y=autocorr.numpy(), marker="o", ax=ax)
ax.axhline(
    autocorr[0].item() / np.e, ls="--", color="crimson", label="1/e of lag-1"
)
ax.axvline(
    WINDOW, ls=":", color="steelblue", label=f"Coherence length = {WINDOW}"
)
ax.set(
    title="Trajectory Coherence",
    xlabel="Lag (tokens apart)",
    ylabel="Hidden-state autocorrelation",
)
ax.legend()
save_figure(RESULTS_DIR / "figure-01.png")


In [ ]:
# Score chunks derived from WINDOW
# Score: cosine_distance_from one window to the next 

In [ ]:
# Course completion: Restore scoring functions after the original scoring note.
def score_chunks(hidden, w):
    """Score chunks in hidden states.

    Parameters
    ----------
    hidden : torch.Tensor
        Trajectory hidden states, shape (num_token, num_dim)
    w : int
        Window size

    Returns
    --------
    torch.Tensor
        Boundary scores (cosine distances)
    """
    seq_len = hidden.shape[0]

    scores = torch.zeros(seq_len)
    for i in range(seq_len):
        start = max(0, i - w)
        end = min(seq_len, i + w)

        left = hidden[start:i]
        right = hidden[i:end]
        if len(left) == 0 or len(right) == 0:
            continue

        L = left.mean(dim=0)
        R = right.mean(dim=0)

        scores[i] = 1.0 - F.cosine_similarity(L, R, dim=0)

    return scores

def scale(x, eps=1e-8):
    """Center and scale a 1-D signal into robust standard units.

    This function works like a z-score, but it uses the median and median
    absolute deviation (MAD) instead of mean and standard deviation.

    Parameters
    ----------
    x : torch.Tensor or array-like
        Signal to standardize
    eps : float
        Epsilon value for numerical stability

    Returns
    -------
    np.ndarray
        Scaled signal
    """
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()

    x = np.asarray(x, dtype=float)
    median = np.median(x)
    mad = median_abs_deviation(x, scale=1.0)

    return (x - median) / (mad + eps)


In [ ]:
# Course completion: Restore boundary detection and trace segmentation.
raw_scores = score_chunks(hidden, w=WINDOW)
scores = scale(raw_scores)

print("Scored", len(scores), "positions")


In [ ]:
# Course completion: Restore boundary detection and trace segmentation.
is_finite = scores[np.isfinite(scores)]
HEIGHT = np.percentile(is_finite, 90)
DISTANCE = 2 * WINDOW

print("Keeping peaks above the 90th percentile: height >=", round(HEIGHT, 2))
print("Peaks must be spaced at least", DISTANCE, "tokens apart")


In [ ]:
# Course completion: Restore boundary detection and trace segmentation.
boundaries, props = find_peaks(scores, distance=DISTANCE, height=HEIGHT)
print("Found", len(boundaries), "boundaries")


In [ ]:
# Course completion: Restore boundary detection and trace segmentation.
cut_points = [0, *boundaries.tolist(), len(cot_ids)]
zipped = zip(cut_points[:-1], cut_points[1:])

for step, (lo, hi) in enumerate(zipped, start=1):
    segment = tokenizer.decode(cot_ids[lo:hi])
    print(f"--- step {step} (tokens {lo}-{hi}) ---")
    print(segment)
    print()


In [ ]:
# Course completion: Restore boundary detection and trace segmentation.
scores_df = pd.DataFrame({"position": np.arange(len(scores)), "score": scores})


In [ ]:
# Course completion: Restore boundary detection and trace segmentation.
fig, ax = plt.subplots()
sns.lineplot(scores_df, x="position", y="score", ax=ax)
ax.scatter(
    boundaries,
    scores[boundaries],
    color="crimson",
    zorder=5,
    label="boundary",
)
ax.set(
    title=f"Boundary Scores (Layer={LAYER}, Window={WINDOW})",
    xlabel="Token position in reasoning trace",
    ylabel="Trajectory change",
)
ax.legend()
save_figure(RESULTS_DIR / "figure-02.png")


In [ ]:
# Course completion: Restore mean-pooling for each segment inline.
def segment_representations(hidden, cut_points):
    """Produce a mean-pooled representation for every reasoning step.

    Parameters
    ----------
    hidden : torch.Tensor
        Trajectory hidden states, shape (num_token, num_dim)
    cut_points : list[int]
        Step boundaries

    Returns
    -------
    tuple[torch.Tensor, list]
        Mean-pooled step vectors and the segmented token spans
    """
    spans = list(zip(cut_points[:-1], cut_points[1:]))

    vecs = []
    for lo, hi in spans:
        vecs.append(hidden[lo:hi].mean(dim=0))

    return torch.stack(vecs), spans


In [ ]:
# Course completion: Restore segment clustering and inspection.
step_vecs, spans = segment_representations(hidden, cut_points)
step_vecs = F.normalize(step_vecs, dim=-1).detach().cpu().numpy()


In [ ]:
# Course completion: Restore segment clustering and inspection.
num_step = len(step_vecs)
if num_step < 3:
    raise ValueError("At least three segments are needed for silhouette selection. Use a longer trace.")

max_k = min(5, num_step - 1)
candidate_k = range(2, max_k + 1)

scores_by_k = {}
for k in candidate_k:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=5167)
    labels_k = kmeans.fit_predict(step_vecs)
    scores_by_k[k] = silhouette_score(step_vecs, labels_k)


In [ ]:
# Course completion: Restore segment clustering and inspection.
best_k = max(scores_by_k, key=lambda k: (scores_by_k[k], k))

for k, score in scores_by_k.items():
    marker = ""
    if k == best_k:
        marker = "  <- best"

    print(f"k={k}: silhouette={score:.3f}{marker}")

kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=5167).fit(step_vecs)


In [ ]:
# Course completion: Restore segment clustering and inspection.
cluster_df = pd.DataFrame(spans, columns=["lo", "hi"])
cluster_df["step"] = np.arange(len(cluster_df))
cluster_df["label"] = kmeans.labels_


In [ ]:
# Course completion: Restore segment clustering and inspection.
cluster_df["text"] = cluster_df.apply(
    lambda row: tokenizer.decode(cot_ids[row["lo"]:row["hi"]]),
    axis=1
)


In [ ]:
# Course completion: Restore segment clustering and inspection.
for c in range(best_k):
    group = cluster_df[cluster_df["label"] == c]

    print(f"=== mode {c}  |  {len(group)} steps ===")
    for _, row in group.iterrows():
        text = repr(row["text"][:100])
        print(f"  step {row['step']:2d}: {text}")

    print()


In [ ]:
# Course completion: Restore segment clustering and inspection.
num_col = int(np.ceil(np.sqrt(num_step)))
num_row = int(np.ceil(num_step / num_col))

grid = np.full((num_row, num_col), np.nan)
grid.flat[:num_step] = kmeans.labels_

cmap = sns.color_palette("deep", best_k)

fig, ax = plt.subplots()
sns.heatmap(
    grid,
    mask=np.isnan(grid),
    cmap=cmap,
    vmin=0,
    vmax=best_k - 1,
    cbar=False,
    square=True,
    linewidths=2,
    xticklabels=False,
    yticklabels=False,
    ax=ax,
)

for i, color in enumerate(cmap):
    ax.scatter([], [], marker="s", s=100, color=color, label=f"mode {i}")

ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
ax.set(
    title="Reasoning Modes Across the Trace",
    xlabel="",
    ylabel="",
)
save_figure(RESULTS_DIR / "figure-03.png")


In [ ]:
# Course completion: Save trace completion, model device, and precision with the analysis.
(RESULTS_DIR / "reasoning_trace.txt").write_text(tokenizer.decode(cot_ids), encoding="utf-8")
trace_record = {"generation_budget": 2048, "generated_tokens": len(ids),
                "reasoning_complete": think_close in ids, "analyzed_positions": len(scores),
                "segments": len(cluster_df), "clusters": int(best_k),
                "model_device": str(model.device), "model_dtype": str(model.dtype)}
(RESULTS_DIR / "trace_status.json").write_text(json.dumps(trace_record, indent=2), encoding="utf-8")


In [ ]:
scores_df.to_csv(RESULTS_DIR / "scores_df.csv", index=True)
cluster_df.to_csv(RESULTS_DIR / "cluster_df.csv", index=True)
observed = {name: module.__version__ for name, module in sys.modules.items()
            if name in {"numpy", "pandas", "torch", "transformers", "sklearn", "scipy", "matplotlib", "seaborn"}}
environment = {"python": platform.python_version(), "imported_packages": observed}
(RESULTS_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print("Completed notebook; results:", RESULTS_DIR)
